In [2]:
import cv2
from paho.mqtt import client as mqtt_client
import numpy as np
import tools
import random
from yolox.tracker.byte_tracker import BYTETracker
import argparse
import pandas as pd
import torch
import ast
import re


/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Beispielwerte für die Argumente des Trackers
tracker_args = argparse.Namespace(
    track_thresh=0.29,       # Beispielwert für den Tracking-Schwellenwert
    track_buffer=15,        # Beispielwert für den Puffer
    mot20=False,            # Beispielwert für MOT20
    match_thresh=0.9        # Beispielwert für den Matching-Schwellenwert
)

purpose = "testing"
video_path = "input/Test.mp4"
csv = 'input/lang_sam.csv' #"input/output_data.csv"
csv2 = 'input/oneformer.csv'  
output_path = "output/wheeltracker_output.mp4"

tracker = BYTETracker(tracker_args)

if purpose == "testing":
    # Read the video file
    cap = cv2.VideoCapture(video_path)
    # Einlesen der CSV-Datei
    data = pd.read_csv(csv, sep=",") #sep ändern, falls nötig
    data2 = pd.read_csv(csv2, sep=",") #für die Interpolation
    

    # Videoeigenschaften abrufen
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec für das Ausgabevideo
    img_size = (width, height)
    frame_max = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video total frames:{frame_max}")
    out = cv2.VideoWriter(output_path, fourcc, fps, img_size)
    
    frame_count = 1 #508

    # Debugging: Überprüfen von img_size
    print(f"img_size: {img_size}, Typ: {type(img_size)}")
    print(f"width: {width}, Typ: {type(width)}")
    print(f"height: {height}, Typ: {type(height)}")

    while cap.isOpened():
        ret, frame = cap.read()
        
        if not ret or frame is None:
            print("End of video or unable to read the frame.")
            break
        if frame_count >= frame_max:
            print("All data processed.")
            break
        value = data.iloc[frame_count - 1, 1]
        value_count = data.iloc[frame_count - 1, 0]

        print(f"Value at frame {value_count}: {value}")
        if isinstance(value, str):
            skip = value.lower() == 'nan' or value.lower() == 'n/a'
        else:
            skip = pd.isna(value)

        if not skip:
            print(str(data.iloc[frame_count + 1, 1]))
            print("Frame: ", frame_count)
            #confidence = data.iloc[i, 1]
            #x_min = data.iloc[i, 2]
            #y_min = data.iloc[i, 3]
            #x_max = data.iloc[i, 4]
            #y_max = data.iloc[i, 5]
            row = data[data['Iteration'] == frame_count].iloc[0]
            if isinstance(row['Label'], (list, np.ndarray)):
                labels = list(row['Label'])
            else:
                labels = ast.literal_eval(row['Label'])

            conf_str = row['Confidence Score']
            if isinstance(conf_str, str):
                # Ersetze alle Whitespaces zwischen Zahlen durch Kommas
                conf_str = re.sub(r'(?<=\d)\s+(?=\d)', ', ', conf_str)
                confidences = ast.literal_eval(conf_str)
            else:
                confidences = ast.literal_eval(row['Confidence Score'])
            confidences = [conf + 0.3 for conf in confidences]
            print(f"Labels: {labels}, Confidences1: {confidences}")
            x_min = ast.literal_eval(row['x_min'])
            y_min = ast.literal_eval(row['y_min'])
            x_max = ast.literal_eval(row['x_max'])
            y_max = ast.literal_eval(row['y_max'])
            for label, conf, xmin, ymin, xmax, ymax in zip(labels, confidences, x_min, y_min, x_max, y_max):
                print(f"Label: {label}, Confidence: {confidences}")
                #bbox = []
                bbox = [xmin, ymin, xmax, ymax]
                print(bbox)
                bbox_and_confidence = torch.tensor([[*bbox, conf]])
                shape = bbox_and_confidence.shape[1]
                print(f"Shape: {shape}")
                img_info = (frame.shape[1], frame.shape[0])
                scale = min(img_size[0] / float(frame.shape[1]), img_size[1] / float(frame.shape[0]))
                print(f"Scale: {scale}")
                online_targets = tracker.update(bbox_and_confidence, img_info, img_size, label)
                # print(f"Tracked Stracks: {len(online_targets.tracked_stracks)}")
                # print(f"Lost Stracks: {len(online_targets.self.lost_stracks)}")
                # print(f"Detections: {len(online_targets.detections)}")
                print(f"Online targets: {online_targets}")
                # online_targets enthält eine Liste von Objekten mit Bounding Boxes und IDs
                for target in online_targets:
                    # Extrahiere die Bounding Box und die ID
                    print(target.track_id, target.label)
                    print(f"Label:{label}")  # ID des Tracks

                    # Konvertieren der Bounding Box in (x1, y1, x2, y2)
                    bbox = target.tlwh
                    x1, y1, w, h = bbox
                    x2, y2 = int(x1 + w), int(y1 + h)
                    x1, y1 = int(x1), int(y1)

                    # Zeichnen der Bounding Box
                    color = (0, 255, 0)  
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

                    # Zeichnen der Track-ID
                    text = f"ID: {target.label} (OT_{target.track_id})"
                    cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 2, color, 2)

            del labels, confidences


            
            out.write(frame)
            frame_count += 1
        else:
            out.write(frame)
            frame_count += 1

    cap.release()
    out.release()

Video total frames:1834
img_size: (1440, 1080), Typ: <class 'tuple'>
width: 1440, Typ: <class 'int'>
height: 1080, Typ: <class 'int'>
Value at frame 1: nan
Value at frame 2: nan
Value at frame 3: nan
Value at frame 4: nan
Value at frame 5: nan
Value at frame 6: nan
Value at frame 7: nan
Value at frame 8: nan
Value at frame 9: nan
Value at frame 10: nan
Value at frame 11: nan
Value at frame 12: nan
Value at frame 13: nan
Value at frame 14: nan
Value at frame 15: nan
Value at frame 16: nan
Value at frame 17: nan
Value at frame 18: nan
Value at frame 19: nan
Value at frame 20: nan
Value at frame 21: nan
Value at frame 22: nan
Value at frame 23: nan
Value at frame 24: nan
Value at frame 25: nan
Value at frame 26: nan
Value at frame 27: nan
Value at frame 28: nan
Value at frame 29: nan
Value at frame 30: nan
Value at frame 31: nan
Value at frame 32: nan
Value at frame 33: nan
Value at frame 34: nan
Value at frame 35: nan
Value at frame 36: nan
Value at frame 37: nan
Value at frame 38: nan
V

/workspace/ByteTrack/yolox/tracker/byte_tracker.py:183: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at  ../aten/src/ATen/native/IndexingUtils.h:30.)
  dets_second = bboxes[inds_second]
/workspace/ByteTrack/yolox/tracker/byte_tracker.py:187: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at  ../aten/src/ATen/native/IndexingUtils.h:30.)
  scores_second = scores[inds_second]


Value at frame 488: ['wheel', 'wheel']
['wheel', 'wheel']
Frame:  488
Labels: ['wheel', 'wheel'], Confidences1: [0.8525396999999999, 0.7804786699999999]
Label: wheel, Confidence: [0.8525396999999999, 0.7804786699999999]
[74.0949478149414, 421.9284362792969, 92.07012176513672, 450.3475036621094]
Shape: 5
Scale: 1.0
dists [[0.2681216]
 [1.       ]]
dists nach mot20 [[0.37604459]
 [1.        ]]
matches [[0 0]] u_track [1] u_detection []
Online targets: [OT_331_(2-12)]
331 wheel
Label:wheel
Label: wheel, Confidence: [0.8525396999999999, 0.7804786699999999]
[153.497314453125, 393.1644592285156, 179.36318969726562, 433.8558044433594]
Shape: 5
Scale: 1.0
dists [[1.]
 [1.]]
dists nach mot20 [[1.]
 [1.]]
matches [] u_track [0 1] u_detection [0]
Online targets: []
Value at frame 489: ['wheel', 'wheel']
['wheel', 'wheel']
Frame:  489
Labels: ['wheel', 'wheel'], Confidences1: [0.8338836000000001, 0.7666025]
Label: wheel, Confidence: [0.8338836000000001, 0.7666025]
[80.01785278320312, 422.186584472